# RNA-Sequencing for HuR Target Identification and Validation Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR^2 RNA-seq dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.sebs-rntb/fair2.json](https://sen.science/doi/10.71728/senscience.sebs-rntb/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.sebs-rntb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Get the metadata object
metadata = dataset.metadata

# Print key metadata attributes
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Published on: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All entities are referenced by their `@id`.
We begin by inspecting available record sets, then examine their fields and columns.

In [ ]:
# List all record sets and their @id
record_sets = dataset.record_sets
print("Record Sets Available:")
for rs in record_sets:
    print(f"- Name: {rs.name}\n  @id: {rs.id}\n  Description: {getattr(rs, 'description', 'No description')}\n")
    
    # List the fields for each record set
    fields = rs.fields
    print("  Fields:")
    for field in fields:
        print(f"    - Name: {field.name}\n      @id: {field.id}\n      Data Type: {getattr(field, 'data_type', 'Unknown')}\n")
    
    # List the columns for each record set
    columns = rs.columns
    print("  Columns:")
    for column in columns:
        print(f"    - Name: {column.name}\n      @id: {column.id}\n      Data Type: {getattr(column, 'data_type', 'Unknown')}\n")
    print("-")

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis.

Below, we dynamically iterate through all record sets and extract the records using their `@id`. This ensures full reference compliance.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Fields (@id) in DataFrame: {df.columns.tolist()}")
    print(df.head(2))
    print("-")

# Choose a record set to explore further (first one)
chosen_record_set_id = record_set_ids[0]
print(f"\nFirst record set to analyze: {chosen_record_set_id}")
df = dataframes[chosen_record_set_id]
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's:
- Filter records based on a numeric field and threshold
- Normalize the field
- Group the records (if possible) by a categorical field

*Note: Make sure to use only the field/column `@id`s as reference in code.*

In [ ]:
# Find numeric fields (@id) from the chosen record set
rs = [r for r in dataset.record_sets if r.id == chosen_record_set_id][0]
numeric_fields = [col.id for col in rs.columns if getattr(col, 'data_type', '').lower() in ['float', 'integer', 'number']]

if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field @id: {numeric_field_id} for filtering and normalization.")
else:
    numeric_field_id = df.select_dtypes(include=['float64', 'int64']).columns.tolist()[0] if not df.empty else None
    print(f"Fallback numeric field: {numeric_field_id}")

threshold = 10

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
normalized_col = f"{numeric_field_id}_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, normalized_col]].head())

# Find possible group field (@id)
categorical_columns = [col.id for col in rs.columns if getattr(col, 'data_type', '').lower() == 'text']
group_field_id = categorical_columns[0] if categorical_columns else df.select_dtypes(include=['object']).columns.tolist()[0]
print(f"Attempting group by field @id: {group_field_id}")

if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

We'll plot:
- Histogram of the numeric field
- Boxplot grouped by a categorical field


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Plot boxplot grouped by the group field
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
    plt.title(f"{numeric_field_id} grouped by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded FAIR-compliant RNA-seq cutoff tables via Croissant schema using the `mlcroissant` library
- Explored record sets and fields using their `@id`
- Extracted and analyzed datasets dynamically for EDA
- Filtered, normalized, and grouped records, referencing all entities by `@id`
- Visualized key distributions and relationships

**Key Takeaway:** By referencing entities via `@id`, and leveraging `mlcroissant`, you can dynamically process and reuse FAIR-compliant datasets across projects, enabling reproducible and scalable bioinformatics analyses.
